# Lesson 10 : Hosted Agents in Microsoft Foundry

When you deploy and run your generated agent's code for production, you can, of course, host your agent by yourself.  
However, if you're working in Microsoft Foundry, you can host your agent by publishing in Microsoft Foundry.

Even when you're writing your code with Agent Framework, you can host your agent in Microsoft Foundry, as a **hosted agent**.  
By using hosted agents in Microsoft Foundry, you can handle your agents without having to build various management functionalities - such as, start/stop, upgrading/versioning, monitoring, horizontal scaling, etc.

> Note : The code built by Agent Framework, LangChain, and custom code can be run as a hosted agent. (Workflow agents in Agent Framework can also work as hosted agents.)  
> When it's written with Agent Framework and LangChain, there exists a pre-built wrapper for hosting your agent as a hosted agent in Microsoft Foundry.

In this exercise, we configure and deploy our simple code agent written by Agent Framework as a hosted agent in Microsoft Foundry.

Here I'll show you the outline of steps to run a hosted agent, but please see [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents) for more details.

> Note : This hosted agent is experimented by using azd extension ```azure.ai.agents``` version ```0.1.18-preview```. (This feature is in preview and there's several restrictions as the time of writing.)

## 1. Prepare environment

To start this exercise, you should install ```azd``` command (Azure Developer CLI) on your working environment.

> Note : Here we use ```azd``` command to configure infrastructure, but you can also manually provision Azure infrastructure and deploy your hosted agent with Python SDK, without ```azd``` command.

After installation, login to Azure in ```azd``` by running the following command.

```
azd auth login --use-device-code
```

Next, install ai agent extension (```azure.ai.agents```) in ```azd``` by running the following command.

```
azd extension install azure.ai.agents
```

> Note : This extension can also be automatically installed by running the following ```azd init -t https://github.com/Azure-Samples/azd-ai-starter-basic``` (initialization by the template), but here we manually install.

Finally, install docker, because hosted agents run on container app as a base.  
In my case, I have used the following command to install docker in Ubuntu 24.04, and logout/login after installation.

```
sudo apt-get -y update
sudo apt-get -y install apt-transport-https ca-certificates curl software-properties-common
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get -y update
apt-cache policy docker-ce
sudo apt-get -y install docker-ce
sudo usermod -aG docker $USER  # Logout and Login to take effect !
```

## 2. Prepare resources in Azure

Please create a Foundry resource in the region where hosted agents are supported.  
For the supported regions by the hosted agent, please see [here](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents#region-availability).

> Note : In this exercise, we prepare Foundry resource in advance, but you can also create Foundry resource in the following provisioning phase, without creating Foundry resources manually by yourself.

In Foundry Portal, deploy an Azure OpenAI model which is supported in Azure OpenAI Responses API. (See [here](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry&tabs=python-key#model-support) for the supported models.)  
In the following setting, we assume we have deployed **gpt-5**.

In hosted agents, other resources (such as, container registry, etc) are also required. In this example, however, these other resources are automatically created by running the following provisioning steps. (No other resources, therefore, should be prepared here.)

## 3. Prepare files

Hosted agents use docker container as its base technology.  
In order for provisioning as a hosted agent, you should prepare the following files in this example.

- Python source code (main.py)
- Python package list (requirements.txt)
- Docker file (Dockerfile)
- Agent configuration (agent.yaml)

In this example, we save these files in ```hosted_agent_work/script``` folder, and we create this folder as follows.

In [1]:
import os
script_folder = os.path.join("hosted_agent_work", "script")
os.makedirs(script_folder, exist_ok=True)

First we prepare Python source code (main.py).

**Currently (Apr 2026), the latest version (1.0.0) of Agent Framework conflicts with ```azure-ai-agentserver-agentframework```**, and here we use old version and ```AzureOpenAIChatClient``` client, instead of ```FoundryChatClient```. (Because ```FoundryChatClient``` cannot be used in this version.)

In this example, we change code in Lesson 1 as follows in order to deploy this agent as a hosted agent in Microsoft Foundry.

- As I have mentioned above, here we use ```AzureOpenAIChatClient```, which is backed by Azure OpenAI API.
- The agent is abstracted by adapter function, ```from_agent_framework()```.
- To get credential, I have changed ```AzureCliCredential``` to ```DefaultAzureCredential```. (Because Managed Identity might be used in hosted agents.)

The ```%%writefile``` directive (first line in the following code) in Jupyter notebook cell indicates that this code is saved as file, not executed here.

In [2]:
%%writefile hosted_agent_work/script/main.py
from typing import Annotated
from pydantic import Field
from random import randint
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework import Agent, tool
from azure.identity import DefaultAzureCredential
from azure.ai.agentserver.agentframework import from_agent_framework

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

if __name__ == "__main__":
    credential = DefaultAzureCredential()
    client = AzureOpenAIChatClient(credential=credential)
    agent = Agent(
        client=client,
        instructions="You are an agent about weather information.",
        tools=[get_weather, get_temperature]
    )
    from_agent_framework(agent).run()

Writing hosted_agent_work/script/main.py


Next we prepare requirements.txt, a list of Python packages, which are installed on docker image generation.

To use adapter for hosted agent (i.e., ```from_agent_framework()``` in above code), we should install Python package ```azure-ai-agentserver-agentframework```.

As I have mentioned above, currently we should specify old version of Agent Framework, because of the conflicts.

In [3]:
%%writefile hosted_agent_work/script/requirements.txt
azure-ai-agentserver-agentframework==1.0.0b17
agent-framework-azure-ai==1.0.0rc3
agent-framework==1.0.0rc3
agent-framework-core==1.0.0rc3

Writing hosted_agent_work/script/requirements.txt


Now we create docker file, ```Dockerfile```.  
The above ```from_agent_framework()``` automatically creates a REST endpoint by port 8088 and we then expose this port in docker image generation as follows.

In [4]:
%%writefile hosted_agent_work/script/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then \
        pip install -r requirements.txt; \
    else \
        echo "No requirements.txt found"; \
    fi

EXPOSE 8088

CMD ["python", "main.py"]

Writing hosted_agent_work/script/Dockerfile


The file, agent.yaml, includes configuration settings - such as, agent name, environment variables, etc. These settings are used in the following provisioning phase by ```azd``` command execution.

As you saw above, we use ```AzureOpenAIChatClient``` in this example, and this client then requires ```AZURE_OPENAI_ENDPOINT``` and ```AZURE_OPENAI_CHAT_DEPLOYMENT_NAME```.  
In this example, therefore, we set these variables as environment variables as follows. (You can also pass these variables by ```.env``` file, instead.)

> Note : In the following yaml configuration, you can see ```${AZURE_OPENAI_ENDPOINT}```, but you don't need to set this environment variable manually by yourself. This will be automatically set up in the following "```azd ai agent init --project-id [PROJECT-RESOURCE-ID]```" command.

**Please replace the following "```gpt-5```" with your setting.**

In [5]:
%%writefile hosted_agent_work/script/agent.yaml
name: hosted-test-agent01
description: This is a hosted agent for demo.
metadata:
  authors:
    - Workshop demo
template:
  name: hosted-test-agent01
  kind: hosted
  protocols:
    - protocol: responses
      version: v1
  environment_variables:
    - name: AZURE_OPENAI_ENDPOINT
      value: ${AZURE_OPENAI_ENDPOINT}
    - name: AZURE_OPENAI_CHAT_DEPLOYMENT_NAME
      value: "{{chat}}"
resources:
  - kind: model
    id: gpt-5
    name: chat

Writing hosted_agent_work/script/agent.yaml


## 4. Provision, deploy, and run

Now the assets are all ready.  
We now start to complete configurations, provision Azure infrastructure, and deploy your hosted agent, with ```azd``` commands.

**All the following settings should be performed in console (terminal)**, not in Jupyter notebook. (Because Jupyter notebook cannot handle the interactive session.)

Before starting, let's change your working directory to subfolder ```hosted_agent_work```. (Because the execution commands automatically detect the files in your directory.)

```bash
cd hosted_agent_work
```

---

Run the following command to prepare infrastructure setting.  
Before runnning, **please change the following placeholders** in the following command. To get string for ```--project-id``` option, go to foundry project resource on [Azure Portal](https://portal.azure.com/), open "Resource Management" - "properties" in left-side navigation, and you then find it in "Resource ID".

```bash
azd ai agent init -m script/agent.yaml --project-id /subscriptions/[SUBSCRIPTION-ID]/resourceGroups/[RESOURCE-GROUP-NAME]/providers/Microsoft.CognitiveServices/accounts/[FOUNDRY-RESOURCE-NAME]/projects/[PROJECT-NAME]
```

During running this command, you will be asked to specify existing Azure Container Registry (ACR) server name and Application Insights connection name. But **please set blank**, and it will then automatically create these resources in provisioning steps.

> Note : If you have already connected to these resources (app insights, container registry) in your Foundry resource, do not set blank, because multiple connection to the same category are not allowed in Microsoft Foundry. (The error will be thrown.)

You will also be asked for the following settings, but here we apply the default settings as follows.  
(These provisioning definitions are then written in azure.yaml.)

- container CPU allocation : 0.25 cores
- container memory allocation : 0.5 Gi
- container minimum number of replicas : 0
- container maximum number of replicas : 1

By running this command, the required configurations (bicep configurations, etc) are setup in "```infra```" folder. The environment variables (such as, ```AZURE_AI_PROJECT_ENDPOINT```, ```AZURE_OPENAI_ENDPOINT```, etc) are automatically associated with your Foundry resource in the configuration.

---

**[Optional] This is not mandatory for running this exercise.**

When you run and test your AI agent locally, follow this step.

First, please set environment for running your agent. In this example, we should set ```AZURE_OPENAI_ENDPOINT``` and ```AZURE_OPENAI_CHAT_DEPLOYMENT_NAME```. (For getting Azure OpenAI endpoint, go to Foundry portal.)

```
export AZURE_OPENAI_ENDPOINT=[AZURE-OPENAI-ENDPOINT]
export AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-5
```

Before running, generate new virtual environment in Python and activate. (Because it installs packages written in ```requirements.txt```.)

Run your agent locally by the following command.

```
azd ai agent run hosted-test-agent01
```

Open another console, and send a message by the following command.

```
azd ai agent invoke "Tell me the weather and temperature in Osaka today." --local
```

> Note : Do not enable managed identity in your working VM. (If it's enabled, the managed identity will be used for your current credential and it will fail.)

---

Now let's provision the infrastructure by running the following command.  
After running this command, all required resources - Application Insights, Log Analytics workspace, and Azure Container Registry (ACR) - are provisioned in the same resource group as your Foundry resource. (Also, the container image is built and registered in ACR.)

```bash
azd provision
```

Currently (Jan 2026), you should additionally set the following ```AcrPull``` permission after the above provisioning has completed. (See [here](https://github.com/microsoft-foundry/foundry-samples/issues/454) for this issue.)

- Go to container registry resource.
- Go to "Access control (IAM)".
- Add "```AcrPull```" role to managed identity of Foundry resource (not Foundry project resource).

---

Now all Azure resources are ready.  
Finally we deploy and start our hosted agent in Microsoft Foundry by running the following command.

```bash
azd deploy hosted-test-agent01
```

> Note :  By running ```azd up```, provisioning (```azd provision```) and deployment (```azd deploy```) are both performed.  
> On the other hand, ```azd down``` cleans up all resources in resource group.  
> By using ```azd up``` / ```azd down```, you can quickly build all resources and clean up all resources.

After the hosted agent is successfully deployed, you can see your agent running in Foundry Portal UI. (The hosted agent is labeled "```hosted```" as type.)

---

**[Optional]** You can start / stop your agent in Foundry Portal.  
Or you can also use "```az cognitiveservices agent```" CLI commands to manage your agent as follows. (Run ```az cognitiveservices agent --help``` or see [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents#manage-hosted-agents) for other management commands.)

```bash
# start agent
az cognitiveservices agent start \
  --account-name [FOUNDRY-RESOURCE-NAME] \
  --project-name [FOUNDRY-PROJECT-NAME] \
  --name [AGENT-NAME] \
  --agent-version [AGENT-VERSION]
# stop agent
az cognitiveservices agent stop \
  --account-name [FOUNDRY-RESOURCE-NAME] \
  --project-name [FOUNDRY-PROJECT-NAME] \
  --name [AGENT-NAME] \
  --agent-version [AGENT-VERSION]
```

> Note : Before deploying as hosted agent, I recommend you to run the container in local docker runtime. (Once deployed, the error tracking will become more difficult.)

## 5. Consume your hosted agent

To consume your hosted agent, you can use Playground in Foundry Portal UI, or invoke API.

Hosted agents have API interface exposed by a REST API that is compatible with OpenAI Responses API. (See "```protocol: responses```" in above agent.yaml setting.)  
You can then use Azure AI Projects SDK (```azure-ai-projects```), OpenAI SDK, or raw REST to consume your hosted agent with API. (See [official document](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents#invoke-hosted-agents) for details.)  
The following code invokes your hosted agent by using Azure AI Projects SDK. (Replace the placeholders with your Foundry project, in which your hosted agent is running.)

In [6]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from IPython.display import Markdown, display

PROJECT_ENDPOINT = "https://[FOUNDRY-RESOURCE-NAME].services.ai.azure.com/api/projects/[FOUNDRY-PROJECT-NAME]"
AGENT_NAME = "hosted-test-agent01"

# initialize the client and retrieve the agent
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=AzureCliCredential(),
    allow_preview=True,
)
agent = project_client.agents.get(agent_name=AGENT_NAME)

# get OpenAI client and send a message
# (it's Responses API compliant.)
openai_client = project_client.get_openai_client()
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference"
        },
    },
    input=[{"role": "user", "content": "Tell me the weather and temperature in Osaka today."}],
)

# show result
display(Markdown(response.output_text))

Osaka, Japan: **Cloudy**, **11 °C**.

## 6. Use your hosted agent in Microsoft ecosystem

Once it's deployed as hosted agent, your agent can also be consumed in Microsoft ecosystem - such as, publishing to Agent 365, using in multi-agent workflows on Foundry, etc.

## 7. Stop your hosted agent

When you have done, stop your agent in Foundry Portal UI or invoking CLI command as follows, in order not to consume cost (billing).

```
az cognitiveservices agent stop \
  --account-name [FOUNDRY-RESOURCE-NAME] \
  --project-name [FOUNDRY-PROJECT-NAME] \
  --name [AGENT-NAME] \
  --agent-version [AGENT-VERSION]
```